# 03 — Mô hình Machine Learning trên metadata (MILK10k)

9 mô hình ML cổ điển, chỉ dùng **metadata 34 chiều**: tuổi (chuẩn hoá + cờ thiếu), giới, skin tone, vị trí giải phẫu
(one-hot) và 14 điểm khái niệm MONET (7 cho ảnh clinical, 7 cho ảnh dermoscopy). Val trên **fold 0**, cùng metric ISIC.

| Nhóm | Mô hình |
|---|---|
| Tuyến tính / kernel | Logistic Regression, SVM (RBF) |
| Láng giềng | KNN (k = 25) |
| Bagging | Random Forest, Extra Trees |
| Boosting | HistGradientBoosting, LightGBM, XGBoost, CatBoost |

Mọi mô hình dùng trọng số mẫu √balanced (trừ KNN). **Không cần GPU** (Accelerator: None), chạy vài phút.
Mỗi mô hình tạo `outputs/ml_<model>__fold0/` gồm `train.log`, `metrics.json`, `submission.csv`.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!pip install -q lightgbm xgboost catboost

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

## 3. Train

In [ ]:
!python scripts/train_tabular.py --model all

## 4. Kết quả

In [ ]:
!python scripts/summarize.py --dirs outputs > /dev/null
import pandas as pd
pd.read_csv("outputs/summary/ml_models.csv", index_col=0).style.format("{:.4f}").highlight_max(color="#cfe2ff")